# Search-8-DancingLinks-Csharp : L'algorithme X et Dancing Links de Knuth en C#

**Navigation** : [<< Search-7 (MCTS)](Search-7-MCTS-And-Beyond.ipynb) | [↑ Série Search (C#)](../README.md) | [Search-9 (Linear Programming) >>](Search-9-LinearProgramming.ipynb)

**Jumeau .NET** du notebook Python [Search-8-DancingLinks](Search-8-DancingLinks.ipynb). Port C# from-scratch (EPIC [#4956](https://github.com/jsboige/CoursIA/issues/4956), prong A [#3801](https://github.com/jsboige/CoursIA/issues/3801) — algorithme pur : .NET n'a pas de bibliothèque DLX dominante, l'implémentation from-scratch **est** la leçon pédagogique).

Donald Knuth a inventé l'**Algorithme X** (résolution de la couverture exacte par backtracking) et la structure de données **Dancing Links (DLX)** qui le rend redoutablement efficace. Cette structure exploite une idée géniale : supprimer et restaurer un nœud d'une liste doublement chaînée circulaire sont des opérations **symétriques** en O(1), ce qui accélère dramatiquement le backtracking.

Nous allons : (1) définir la couverture exacte, (2) implanter l'Algorithme X naïf, (3) présenter la structure Dancing Links, (4) implanter DLX en C#, (5) l'appliquer au pavage, (6) comparer ses performances au backtracking classique.

## 1. Le problème de couverture exacte (~15 min)

### Définition formelle

Soit un univers $U = \{1, 2, \dots, n\}$ et une collection $\mathcal{S} = \{S_1, S_2, \dots, S_m\}$ de sous-ensembles de $U$. Trouver une **couverture exacte** = un sous-ensemble $\mathcal{S}^* \subseteq \mathcal{S}$ tel que chaque élément de $U$ est contenu dans **exactement un** $S_i \in \mathcal{S}^*$.

On représente le problème par une **matrice binaire** $A$ ($m$ lignes × $n$ colonnes) où $A[i,j] = 1$ si $j \in S_i$. Une couverture exacte = un ensemble de lignes dont la somme colonne par colonne vaut exactement 1 partout.

Le problème est **NP-complet** : aucune méthode polynomiale connue. Applications : Sudoku, pavage de polyominos, planning de personnel, résolution de casse-têtes (N-reines, pentominoes).

In [1]:
using System.Text;

// Exemple canonique : Univers U = {1,2,3,4,5,6,7}
// S1={1,4,7}, S2={1,4}, S3={4,5,7}, S4={3,5,6}, S5={2,3,6,7}, S6={2,7}
// Solution attendue : {S2, S4, S6} car S2∪S4∪S6 = {1,4}∪{3,5,6}∪{2,7} = U
int[,] matrix = {
    { 1, 0, 0, 1, 0, 0, 1 },  // S1
    { 1, 0, 0, 1, 0, 0, 0 },  // S2
    { 0, 0, 0, 1, 1, 0, 1 },  // S3
    { 0, 0, 1, 0, 1, 1, 0 },  // S4
    { 0, 1, 1, 0, 0, 1, 1 },  // S5
    { 0, 1, 0, 0, 0, 0, 1 },  // S6
};
string[] rowNames = { "S1", "S2", "S3", "S4", "S5", "S6" };
int nRows = matrix.GetLength(0), nCols = matrix.GetLength(1);

var sb = new StringBuilder();
sb.AppendLine("=== Couverture exacte : exemple canonique ===");
sb.AppendLine($"Univers U = {{1,2,3,4,5,6,7}}  (n={nCols} éléments, m={nRows} sous-ensembles)");
sb.AppendLine();
sb.AppendLine("Matrice binaire :");
sb.AppendLine("      1 2 3 4 5 6 7");
for (int i = 0; i < nRows; i++)
{
    var row = new StringBuilder();
    for (int j = 0; j < nCols; j++) row.Append(matrix[i, j] == 1 ? " 1" : " .");
    sb.AppendLine($"  {rowNames[i]} {row}");
}
sb.AppendLine();
// Vérification de la solution {S2, S4, S6} = indices 1, 3, 5
int[] solRows = { 1, 3, 5 };
int[] coverCount = new int[nCols];
foreach (var r in solRows)
    for (int j = 0; j < nCols; j++) coverCount[j] += matrix[r, j];
bool isExact = coverCount.All(c => c == 1);
sb.AppendLine($"Solution proposée : {{S2, S4, S6}}");
sb.AppendLine($"Couverture par colonne : [{string.Join(", ", coverCount)}]");
sb.AppendLine($"Couverture exacte ? {(isExact ? "OUI" : "NON")}  (chaque colonne == 1)");
sb.ToString().Display();

=== Couverture exacte : exemple canonique ===
Univers U = {1,2,3,4,5,6,7}  (n=7 éléments, m=6 sous-ensembles)

Matrice binaire :
      1 2 3 4 5 6 7
  S1  1 . . 1 . . 1
  S2  1 . . 1 . . .
  S3  . . . 1 1 . 1
  S4  . . 1 . 1 1 .
  S5  . 1 1 . . 1 1
  S6  . 1 . . . . 1

Solution proposée : {S2, S4, S6}
Couverture par colonne : [1, 1, 1, 1, 1, 1, 1]
Couverture exacte ? OUI  (chaque colonne == 1)


## 2. L'Algorithme X de Knuth (~15 min)

### Principe

L'Algorithme X est une procédure récursive de backtracking :

1. Si la matrice est vide → **solution trouvée**.
2. Choisir une colonne $c$ (idéalement avec le moins de 1 — heuristique).
3. Pour chaque ligne $r$ ayant un 1 dans $c$ :
   - Inclure $r$ dans la solution partielle.
   - Éliminer toutes les colonnes conflictuelles (celles où $r$ a un 1) et toutes les lignes qui touchent ces colonnes.
   - Récursion sur la matrice réduite.
   - Si échec, annuler et essayer la ligne suivante.

La version **naïve** recopie la matrice à chaque étape (coûteuse). Dancing Links (section 3) fera tout en O(1).

In [2]:
// Algorithme X naïf (recopie la matrice — référence pédagogique)

static List<int>? AlgorithmXNaive(int[,] matrix)
{
    int nRows = matrix.GetLength(0), nCols = matrix.GetLength(1);
    var rowIdx = Enumerable.Range(0, nRows).ToList();
    var sol = new List<int>();
    bool Solve(int[,] m, List<int> rows, List<int> solution)
    {
        int nc = m.GetLength(1);
        if (nc == 0) return true;  // matrice vide = solution
        // Heuristique : colonne avec le minimum de 1
        int bestCol = -1, minCount = int.MaxValue;
        for (int j = 0; j < nc; j++)
        {
            int cnt = 0;
            for (int i = 0; i < m.GetLength(0); i++) cnt += m[i, j];
            if (cnt == 0) return false;  // colonne sans 1 = impossible
            if (cnt < minCount) { minCount = cnt; bestCol = j; }
        }
        for (int i = 0; i < m.GetLength(0); i++)
        {
            if (m[i, bestCol] != 1) continue;
            // Colonnes à couvrir = celles où la ligne i a un 1
            var colsToCover = new List<int>();
            for (int j = 0; j < nc; j++) if (m[i, j] == 1) colsToCover.Add(j);
            // Lignes à garder = celles sans 1 sur aucune colonne conflictuelle
            var rowsToKeep = new List<int>();
            for (int i2 = 0; i2 < m.GetLength(0); i2++)
            {
                bool keep = true;
                foreach (var cc in colsToCover) if (m[i2, cc] == 1) { keep = false; break; }
                if (keep) rowsToKeep.Add(i2);
            }
            // Construire la matrice réduite
            var newRows = rowsToKeep.Select(r => rows[r]).ToList();
            var keptCols = Enumerable.Range(0, nc).Where(j => !colsToCover.Contains(j)).ToList();
            int[,] reduced = new int[rowsToKeep.Count, keptCols.Count];
            for (int a = 0; a < rowsToKeep.Count; a++)
                for (int b = 0; b < keptCols.Count; b++)
                    reduced[a, b] = m[rowsToKeep[a], keptCols[b]];
            solution.Add(rows[i]);
            if (Solve(reduced, newRows, solution)) return true;
            solution.RemoveAt(solution.Count - 1);
        }
        return false;
    }
    return Solve(matrix, rowIdx, sol) ? sol : null;
}

var sw = System.Diagnostics.Stopwatch.StartNew();
var naiveSol = AlgorithmXNaive(matrix);
sw.Stop();
var sb2 = new StringBuilder();
sb2.AppendLine("=== Algorithme X naïf (réduction de matrice) ===");
if (naiveSol != null)
{
    sb2.AppendLine($"Solution trouvée : [{string.Join(", ", naiveSol.Select(r => rowNames[r]))}]");
    sb2.AppendLine("Solution attendue : [S2, S4, S6]");
}
else sb2.AppendLine("Aucune solution trouvée.");
sb2.AppendLine($"Temps : {sw.Elapsed.TotalMilliseconds:F3} ms");
sb2.ToString().Display();

=== Algorithme X naïf (réduction de matrice) ===
Solution trouvée : [S2, S4, S6]
Solution attendue : [S2, S4, S6]
Temps : 3,231 ms



(3,17): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Exercice 1 : Implémenter l'heuristique MRV pour le choix de colonne

L'heuristique **MRV (Minimum Remaining Values)** choisit la colonne avec le **moins de 1** strictement positif. C'est elle que l'Algorithme X utilise implicitement ci-dessus. Isolez-la dans une fonction dédiée — elle retourne l'index de la colonne optimale, ou `-1` si une colonne n'a aucun 1 (échec).

**Indices** : `# Etape 1` compter les 1 par colonne · `# Etape 2` détecter une colonne à 0 → retourner -1 · `# Etape 3` retourner l'index du minimum strictement positif.

In [3]:
static int ChooseColumnMrv(int[,] m)
{
    // TODO etudiant : implementer l'heuristique MRV
    // Etape 1 : compter les 1 par colonne
    // Etape 2 : si une colonne a 0 choix, retourner -1
    // Etape 3 : trouver la colonne avec le minimum positif
    return -1;  // TODO etudiant : remplacer par l'implementation
}

"Exercice a completer : heuristique MRV pour le choix de colonne".Display();

Exercice a completer : heuristique MRV pour le choix de colonne

### Exercice 2 : Vérifier une solution de couverture exacte

Étant donnée une matrice binaire et une liste d'indices de lignes, écrire une fonction `IsExactCover` qui retourne `true` ssi ces lignes forment une couverture exacte (chaque colonne est couverte exactement une fois).

**Indices** : `# Etape 1` extraire et sommer les lignes sélectionnées · `# Etape 2` vérifier que chaque colonne vaut exactement 1.

In [4]:
static bool IsExactCover(int[,] m, List<int> solutionRows)
{
    // TODO etudiant : implementer la verification
    // Etape 1 : extraire les lignes de la solution
    // Etape 2 : sommer les colonnes
    // Etape 3 : verifier que chaque colonne a exactement un 1
    return false;  // TODO etudiant : remplacer par la verification
}

"Exercice a completer : verification d'une couverture exacte".Display();

Exercice a completer : verification d'une couverture exacte

## 3. Dancing Links — la structure de données (~20 min)

### L'idée géniale de Knuth

Dancing Links repose sur un fait trivial mais puissant des **listes doublement chaînées circulaires** :

```
// Supprimer un nœud X de sa liste (horizontale) :
X.R.L = X.L;
X.L.R = X.R;

// Restaurer X exactement (opération inverse, O(1)) :
X.L.R = X;
X.R.L = X;
```

Comme chaque nœud « connaît » ses voisins, **supprimer puis restaurer est symétrique**. Pas besoin de pile de undo, pas de recopie : le backtracking devient quasiment gratuit.

On organise la matrice binaire comme un **réseau toroïdal** de nœuds :
- Chaque **1** de la matrice = un `DataNode`.
- Chaque **colonne** a un `ColumnNode` (tête) qui compte ses 1 (`size`).
- Un `Header` (racine « ROOT ») chaîne toutes les `ColumnNode`.
- Chaque nœud a **4 pointeurs** : `L` (left), `R` (right), `U` (up), `D` (down), plus `C` (sa colonne).
- Toutes les listes sont **circulaires** (le dernier pointe vers le premier).

**Opération `cover(c)`** : retire la colonne `c` de la liste des colonnes, puis retire chaque ligne qui contient un 1 dans `c` (en parcourant les autres colonnes de cette ligne et en les détachant verticalement). **`uncover(c)`** fait exactement l'inverse, dans l'ordre inverse — crucial pour restaurer l'état.

In [5]:
// Structure de données Dancing Links (implémentation C# from-scratch)

public class DlxNode
{
    public string Name = "";     // nom de la colonne (pour ColumnNode)
    public int Size;             // nombre de 1 dans la colonne (ColumnNode seulement)
    public int RowIndex = -1;    // index de ligne (DataNode seulement)
    public DlxNode L, R, U, D, C;  // 4 pointeurs + colonne
    public bool IsColumn => RowIndex == -1 && C == this;
}

static DlxNode MakeColumn(string name)
{
    var c = new DlxNode { Name = name };
    c.L = c.R = c.U = c.D = c.C = c;  // circulaire sur soi-même
    return c;
}

static DlxNode MakeDataNode(DlxNode column, int rowIndex)
{
    var n = new DlxNode { C = column, RowIndex = rowIndex };
    n.L = n.R = n.U = n.D = n;
    return n;
}

// cover : détache la colonne et toutes ses lignes conflictuelles
static void Cover(DlxNode column)
{
    column.L.R = column.R;
    column.R.L = column.L;
    for (var i = column.D; i != column; i = i.D)
        for (var j = i.R; j != i; j = j.R)
        {
            j.D.U = j.U;
            j.U.D = j.D;
            j.C.Size--;
        }
}

// uncover : restaure exactement (ordre inverse — crucial)
static void Uncover(DlxNode column)
{
    for (var i = column.U; i != column; i = i.U)
        for (var j = i.L; j != i; j = j.L)
        {
            j.C.Size++;
            j.D.U = j;
            j.U.D = j;
        }
    column.L.R = column;
    column.R.L = column;
}

// Construction de la structure toroïdale depuis une matrice binaire
static DlxNode BuildDlxStructure(int[,] matrix, out List<DlxNode> rowHeads)
{
    int nRows = matrix.GetLength(0), nCols = matrix.GetLength(1);
    var header = MakeColumn("ROOT");
    var columns = new DlxNode[nCols];
    var prevCol = header;
    for (int j = 0; j < nCols; j++)
    {
        columns[j] = MakeColumn(j.ToString());
        columns[j].L = prevCol;
        prevCol.R = columns[j];
        prevCol = columns[j];
    }
    header.L = columns[nCols - 1];
    columns[nCols - 1].R = header;
    rowHeads = new List<DlxNode>();
    for (int r = 0; r < nRows; r++)
    {
        DlxNode firstInRow = null, prevInRow = null;
        for (int j = 0; j < nCols; j++)
        {
            if (matrix[r, j] != 1) continue;
            var node = MakeDataNode(columns[j], r);
            // insertion verticale en bas de la colonne
            node.U = columns[j].U;
            node.D = columns[j];
            columns[j].U.D = node;
            columns[j].U = node;
            columns[j].Size++;
            // chaînage horizontal
            if (firstInRow == null) firstInRow = node;
            if (prevInRow != null) { node.L = prevInRow; prevInRow.R = node; }
            prevInRow = node;
        }
        if (firstInRow != null) { firstInRow.L = prevInRow; prevInRow.R = firstInRow; rowHeads.Add(firstInRow); }
    }
    return header;
}

"Classes Dancing Links (DlxNode) + Cover/Uncover + BuildDlxStructure definies.".Display();

Classes Dancing Links (DlxNode) + Cover/Uncover + BuildDlxStructure definies.

## 4. Implémentation DLX (~25 min)

L'algorithme DLX = l'Algorithme X **opérant sur la structure Dancing Links**. À chaque étape :

1. Si `header.R == header` → plus de colonnes → **solution trouvée**.
2. Choisir la colonne `c` de `size` minimum (heuristique S — équivalente à MRV).
3. `cover(c)`.
4. Pour chaque nœud `r` dans la colonne `c` (descendre) :
   - Ajouter `r` à la solution.
   - `cover` de chaque colonne touchée par la ligne de `r`.
   - Récursion. Si succès → retour vrai.
   - Sinon : `uncover` dans l'ordre inverse, retirer `r` de la solution.
5. `uncover(c)` et retourner faux.

La magie : `cover`/`uncover` sont O(1) par nœud, donc le backtracking ne recopie **aucune matrice**.

In [6]:
// Algorithme X avec Dancing Links (DLX) — récursif, heuristique S
static bool Search(DlxNode header, List<DlxNode> solution)
{
    if (header.R == header) return true;  // plus de colonnes = solution
    // Choisir la colonne avec le minimum de 1 (heuristique S)
    DlxNode c = null;
    int minSize = int.MaxValue;
    for (var j = header.R; j != header; j = j.R)
        if (j.Size < minSize) { minSize = j.Size; c = j; }
    if (minSize == 0) return false;  // colonne vide = cul-de-sac
    Cover(c);
    for (var r = c.D; r != c; r = r.D)
    {
        solution.Add(r);
        for (var j = r.R; j != r; j = j.R) Cover(j.C);
        if (Search(header, solution)) return true;
        // backtrack
        solution.RemoveAt(solution.Count - 1);
        for (var j = r.L; j != r; j = j.L) Uncover(j.C);
    }
    Uncover(c);
    return false;
}

static List<int>? SolveDlx(int[,] matrix)
{
    var header = BuildDlxStructure(matrix, out _);
    var sol = new List<DlxNode>();
    if (Search(header, sol)) return sol.Select(n => n.RowIndex).ToList();
    return null;
}

var swDlx = System.Diagnostics.Stopwatch.StartNew();
var dlxSol = SolveDlx(matrix);
swDlx.Stop();
var sb3 = new StringBuilder();
sb3.AppendLine("=== Algorithme X avec Dancing Links (DLX) ===");
if (dlxSol != null)
{
    sb3.AppendLine($"Solution DLX : [{string.Join(", ", dlxSol.Select(r => rowNames[r]))}]");
    sb3.AppendLine("Solution attendue : [S2, S4, S6]");
    sb3.AppendLine($"Vérification : {(dlxSol.Count == 3 && dlxSol.Contains(1) && dlxSol.Contains(3) && dlxSol.Contains(5) ? "CORRECT" : "?" )}");
}
else sb3.AppendLine("Aucune solution trouvée.");
sb3.AppendLine($"Temps : {swDlx.Elapsed.TotalMilliseconds:F3} ms");
sb3.ToString().Display();

=== Algorithme X avec Dancing Links (DLX) ===
Solution DLX : [S2, S4, S6]
Solution attendue : [S2, S4, S6]
Vérification : CORRECT
Temps : 1,697 ms



(25,17): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Exercice 3 : Compter toutes les solutions

La fonction `SolveDlx` ci-dessus s'arrête à la **première** solution. Écrivez `CountAllSolutions` qui compte **toutes** les couvertures exactes d'une matrice. Indice : la structure `Cover`/`Uncover` symétrique permet d'énumérer sans dupliquer ni Corrompre l'état.

**Indices** : `# Etape 1` construire la structure DLX · `# Etape 2` modifier `Search` pour qu'au cas de base (`header.R == header`) elle incrémente un compteur et continue au lieu de retourner vrai · `# Etape 3` retourner le total.

In [7]:
static int CountAllSolutions(int[,] matrix)
{
    // TODO etudiant : compter toutes les solutions
    // Etape 1 : construire la structure DLX avec BuildDlxStructure
    // Etape 2 : modifier Search pour compter au lieu de s'arreter a la premiere
    // Etape 3 : lancer la recherche exhaustive et retourner le compteur
    return 0;  // TODO etudiant : remplacer par le vrai comptage
}

"Exercice a completer : comptage de toutes les solutions".Display();

Exercice a completer : comptage de toutes les solutions

## 5. Application — Pavage de polyominos (~10 min)

Le **pavage** est l'application vedette de DLX : paver une grille avec des formes (polyominos) revient à une couverture exacte.

- **Colonnes** = une par case de la grille (chaque case doit être couverte exactement une fois).
- **Lignes** = un **placement** (forme + position + rotation) : la ligne a un 1 dans la case occupée par chaque cellule du placement.

Résoudre la couverture exacte = trouver un jeu de placements non conflictuels couvrant toute la grille.

### Exemple : grille 3×2 pavée par des triominos

Deux formes : le triomino **I** (3 cases en ligne) et le triomino **L**. Construisons la matrice et résolvons-la avec DLX.

In [8]:
// Pavage d'une grille 3×2 par deux triominos (I vertical, L)
// Représentation : cellule (ligne, col) → index = ligne * largeur + col

static (int[,], List<string>) BuildTilingMatrix(int gridW, int gridH, Dictionary<string, List<(int,int)>> polyominos)
{
    var placements = new List<(string name, List<(int r,int c)> cells)>();
    foreach (var (polyName, shape) in polyominos)
    {
        // Normaliser la forme
        int minR = shape.Min(p => p.Item1), minC = shape.Min(p => p.Item2);
        var norm = shape.Select(p => (p.Item1 - minR, p.Item2 - minC)).ToList();
        int sh = norm.Max(p => p.Item1) + 1, sw = norm.Max(p => p.Item2) + 1;
        // Essayer les 4 rotations
        var current = norm;
        int ch = sh, cw = sw;
        for (int rot = 0; rot < 4; rot++)
        {
            // Essayer chaque position dans la grille
            for (int sr = 0; sr <= gridH - ch; sr++)
                for (int sc = 0; sc <= gridW - cw; sc++)
                {
                    var cells = current.Select(p => (p.Item1 + sr, p.Item2 + sc)).ToList();
                    if (cells.All(p => p.Item1 >= 0 && p.Item1 < gridH && p.Item2 >= 0 && p.Item2 < gridW))
                        placements.Add((polyName, cells));
                }
            // Rotation 90° : (r,c) -> (c, ch-1-r)
            var rot2 = current.Select(p => (p.Item2, ch - 1 - p.Item1)).ToList();
            current = rot2;
            (ch, cw) = (cw, ch);  // les dimensions s'échangeant
        }
    }
    int nConstraints = gridW * gridH;
    int[,] m = new int[placements.Count, nConstraints];
    var labels = new List<string>();
    for (int i = 0; i < placements.Count; i++)
    {
        foreach (var (r, c) in placements[i].cells)
            m[i, r * gridW + c] = 1;
        labels.Add($"{placements[i].name}@{string.Join("|", placements[i].cells.Select(p => $"{p.r},{p.c}"))}");
    }
    return (m, labels);
}

var triominos = new Dictionary<string, List<(int,int)>>
{
    ["I"] = new() { (0,0), (1,0), (2,0) },  // vertical (3×1)
    ["L"] = new() { (0,0), (1,0), (1,1) },  // forme L
};

var (mTiling, labels) = BuildTilingMatrix(2, 3, triominos);  // grille 2×3
var tilingSol = SolveDlx(mTiling);
var sb4 = new StringBuilder();
sb4.AppendLine("=== Pavage d'une grille 2×3 par des triominos ===");
sb4.AppendLine($"Matrice : {mTiling.GetLength(0)} placements × {mTiling.GetLength(1)} cases-contraintes");
if (tilingSol != null)
{
    sb4.AppendLine($"Solution : {tilingSol.Count} placements");
    foreach (var idx in tilingSol) sb4.AppendLine($"  - {labels[idx]}");
}
else sb4.AppendLine("Aucun pavage exact trouvé.");
sb4.ToString().Display();

=== Pavage d'une grille 2×3 par des triominos ===
Matrice : 12 placements × 6 cases-contraintes
Solution : 2 placements
  - I@0,0|1,0|2,0
  - I@0,1|1,1|2,1


## 6. Comparaison de performances DLX vs backtracking (~10 min)

Comparons DLX à un **backtracking classique** (sans la structure Dancing Links, qui re-teste la cohérence à chaque pas) sur une instance de pavage plus large. L'écart illustre pourquoi la structure de données — pas seulement l'algorithme — compte en pratique.

In [9]:
// Backtracking classique sur couverture exacte (référence naive pour la comparaison)
static bool BacktrackExactCover(int[,] m, List<int> partial, ref int bestCount, HashSet<int> coveredCols)
{
    int nCols = m.GetLength(1);
    if (coveredCols.Count == nCols) { bestCount++; return true; }
    // Trouver la première colonne non couverte
    int targetCol = -1;
    for (int j = 0; j < nCols; j++) if (!coveredCols.Contains(j)) { targetCol = j; break; }
    for (int r = 0; r < m.GetLength(0); r++)
    {
        if (m[r, targetCol] != 1) continue;
        // Vérifier la compatibilité (pas de chevauchement)
        bool ok = true;
        for (int j = 0; j < nCols; j++) if (m[r, j] == 1 && coveredCols.Contains(j)) { ok = false; break; }
        if (!ok) continue;
        var newCols = new List<int>();
        for (int j = 0; j < nCols; j++) if (m[r, j] == 1) { coveredCols.Add(j); newCols.Add(j); }
        partial.Add(r);
        if (BacktrackExactCover(m, partial, ref bestCount, coveredCols)) { /* on ne stoppe pas pour mesure */ }
        partial.RemoveAt(partial.Count - 1);
        foreach (var j in newCols) coveredCols.Remove(j);
    }
    return false;
}

// Comparaison sur un pavage 3×3 (instance plus large)
var (mBig, labelsBig) = BuildTilingMatrix(3, 3, triominos);

var swBt = System.Diagnostics.Stopwatch.StartNew();
int btCount = 0;
BacktrackExactCover(mBig, new List<int>(), ref btCount, new HashSet<int>());
swBt.Stop();

var swD = System.Diagnostics.Stopwatch.StartNew();
var bigSol = SolveDlx(mBig);
swD.Stop();

var sb5 = new StringBuilder();
sb5.AppendLine("=== Comparaison DLX vs backtracking classique (pavage 3×3) ===");
sb5.AppendLine($"Taille de la matrice : {mBig.GetLength(0)} placements × {mBig.GetLength(1)} contraintes");
sb5.AppendLine($"DLX              : {swD.Elapsed.TotalMilliseconds:F3} ms  → {(bigSol != null ? "solution trouvée" : "pas de solution")}");
sb5.AppendLine($"Backtracking     : {swBt.Elapsed.TotalMilliseconds:F3} ms");
sb5.AppendLine();
sb5.AppendLine("DLX évite la recopie de matrice (cover/uncover O(1) par nœud),");
sb5.AppendLine("tandis que le backtracking classique re-vérifie la compatibilité à chaque pas.");
sb5.ToString().Display();

=== Comparaison DLX vs backtracking classique (pavage 3×3) ===
Taille de la matrice : 28 placements × 9 contraintes
DLX              : 0,017 ms  → solution trouvée
Backtracking     : 0,455 ms

DLX évite la recopie de matrice (cover/uncover O(1) par nœud),
tandis que le backtracking classique re-vérifie la compatibilité à chaque pas.


## 7. Exemple guidé + Exercice 4

### Exemple guidé : instance simple

Revenons à l'exemple canonique (section 1). La matrice 6×7 admet une unique couverture exacte `{S2, S4, S6}`. Les deux solveurs (`AlgorithmXNaive` et `SolveDlx`) la retrouvent — DLX sans jamais recopier la matrice.

### Exercice 4 : Pavage d'une grille 3×5 avec des trominos

**Énoncé** : On souhaite paver une grille 3×5 (15 cases) avec 5 triominos. Modifiez le dictionnaire `triominos` et la taille de grille dans la cellule de pavage pour résoudre ce problème. Combien de pavages distincts existe-t-il ?

In [10]:
// Exercice 4 : Pavage d'une grille 3×5 avec des triominos
// TODO etudiant : adapter BuildTilingMatrix(5, 3, ...) avec les bonnes formes
// puis compter le nombre de pavages distincts (utiliser CountAllSolutions de l'exercice 3).
"Exercice a completer : pavage d'une grille 3×5 avec des triominos".Display();

Exercice a completer : pavage d'une grille 3×5 avec des triominos

---

## Conclusion

Vous avez implémenté **from-scratch** l'Algorithme X de Knuth avec la structure **Dancing Links (DLX)** en C#, validé sur l'exemple canonique de couverture exacte et appliqué au pavage de polyominos. Leçon centrale : la performance d'un algorithme de backtracking ne dépend pas que de la stratégie de recherche, mais surtout de la **structure de données** — la symétrie cover/uncover en O(1) est ce qui rend DLX si efficace sur les problèmes NP-complets à grande matrice creuse (Sudoku, N-reines, pentominoes).

**Verdict #3801 (prong A — SOTA-OK)** : .NET n'a pas de bibliothèque DLX dominante (contrairement à Python où on pourrait importer un solveur) ; l'implémentation from-scratch de la structure toroïdale et des opérations cover/uncover **est** la leçon pédagogique. Les sorties texte passent par `StringBuilder.Display()` plutôt qu'une lib de plotting — en exécution papermill headless, `Console.WriteLine` est avalé, `.Display()` capture `text/plain` (verdict intrinsèque #3436, cohérent avec le twin GA).

**Navigation** : [<< Search-7 MCTS](Search-7-MCTS-And-Beyond.ipynb) | [↑ Série Search (C#)](../README.md) | [Search-9 Linear Programming >>](Search-9-LinearProgramming.ipynb)